This is a notebook to reproduce parameterisation plots for:

S. von Buelow, K. E. Johansson, K. Lindorff-Larsen. AF-CALVADOS: AlphaFold-guided simulations of multi-domain proteins at the proteome level. Protein Science 2026

The code requires the calvados package (https://github.com/KULL-Centre/CALVADOS) and its dependencies, as well as the packages loaded in #Input.

# Input

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import calvados as cal

import math

import numba as nb

import MDAnalysis as mda

from scipy.stats import spearmanr, pearsonr, entropy, linregress

from tqdm import tqdm
import copy

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [ ]:
plt.rcParams['font.size'] = 6
plt.rcParams['ytick.major.size'] = 2
plt.rcParams['ytick.minor.size'] = 1
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['xtick.minor.size'] = 1
plt.rcParams['xtick.labelsize'] = 6
plt.rcParams['ytick.labelsize'] = 6
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['axes.labelpad'] = 2
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['figure.dpi'] = 300
plt.rcParams['lines.markersize'] = 3
plt.rcParams['lines.markeredgewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 1.
plt.rcParams['font.serif'] = 'Times New Roman'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.monospace'] = 'Courier New'
plt.rcParams['figure.max_open_warning'] = 30

col1x = 3.425 # column
col2x = 7. # 2x column

In [ ]:
def calc_idr(scale, min_scale=0.05):
    high_pass = np.where(scale >= min_scale, 1, 0)
    high_pass = np.sum(high_pass,axis=1)
    idr = np.where(high_pass == 0, 1, 0)
    idr_length = np.sum(idr)
    return idr, idr_length

In [ ]:
def plotmat(ax,matrix,cmap=plt.cm.Blues,vmin=0,vmax=1,
            pad=0.05,size="5%",unitlabel='nm'):
    
    _ = ax.imshow(matrix,cmap=cmap,vmin=vmin,vmax=vmax)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=size, pad=pad)
    plt.colorbar(_,cax=cax,label=unitlabel)

# MDP class

In [ ]:
class MDP:
    def __init__(
        self,
        dataset,
        csv_file=None,
        sim_folder=None,
        pdb_folder=None,
        colabfold=0,
        k_gos=[],
        pae_shifts=[],
        pae_widths=[],
        bfac_shift=0.8,
        bfac_width=50.,
        cutoff_restraint=0.9,
    ):
        self.dataset = dataset

        if sim_folder is None:
            self.sim_folder = f'sims_{self.dataset}'
        else:
            self.sim_folder = sim_folder
        if pdb_folder is None:
            self.pdb_folder = f'pdbs_{self.dataset}'
        else:
            self.pdb_folder = pdb_folder

        self.colabfold = colabfold
        self.df = pd.read_csv(csv_file)
        self.k_gos=k_gos
        self.pae_shifts=pae_shifts
        self.pae_widths=pae_widths
        self.bfac_shift=bfac_shift
        self.bfac_width=bfac_width
        self.cutoff_restraint=cutoff_restraint

    def get_param_string(self,k_go=0.,pae_shift=0.,pae_width=0.):
        param_string = f'k_{k_go:.2f}_cutoff_{self.cutoff_restraint:.2f}_bshift_{self.bfac_shift:.2f}_bwidth_{self.bfac_width:.2f}_paeshift_{pae_shift:.2f}_paewidth_{pae_width:.2f}'
        return param_string

    def calc_rgs(self,min_key=None,max_key=None,
               start=10,stop=None,step=1,
               save=True):
        self.rgs = np.zeros((len(self.pae_widths), len(self.k_gos), len(self.pae_shifts), len(self.df[min_key:max_key])))
        for pw_idx, pae_width in enumerate(self.pae_widths):
            for k_idx, k_go in enumerate(self.k_gos):
                print(pae_width, k_go)
                for ps_idx, pae_shift in enumerate(self.pae_shifts):
                    for d_idx, (key, val) in enumerate(self.df[min_key:max_key].iterrows()):
                        name = val['uniprot']

                        param_string = self.get_param_string(k_go=k_go,pae_shift=pae_shift,pae_width=pae_width)
                        path = f'{self.sim_folder}/sims_{param_string}'
                        u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
                        ag = u.atoms
                        rg = cal.analysis.calc_rg(u, ag, start=start, stop=stop, step=step)
                        self.rgs[pw_idx,k_idx,ps_idx,d_idx] = np.mean(rg)
        if save:
            np.save(f'rgs/rgs_{self.dataset}.npy', self.rgs)

    def load_rgs(self):
        self.rgs = np.load(f'rgs/rgs_{self.dataset}.npy')

    def calc_dmaps(self,min_key=None,max_key=None,
               start=10,stop=None,step=1,
               save=True,manual=True):
        for pw_idx, pae_width in enumerate(self.pae_widths):
            for k_idx, k_go in enumerate(self.k_gos):
                print(pae_width, k_go)
                for ps_idx, pae_shift in enumerate(self.pae_shifts):
                    for d_idx, (key, val) in enumerate(self.df[min_key:max_key].iterrows()):
                        name = val['uniprot']
                        param_string = self.get_param_string(k_go=k_go,pae_shift=pae_shift,pae_width=pae_width)
                        path = f'{self.sim_folder}/sims_{param_string}'
                        u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
                        ag = u.atoms

                        if manual:
                            dmaps_std = self.calc_dmap_std_manual(name,path,start=start,stop=stop,step=step)
                        else:
                            dmaps_std = self.calc_dmap_std(name,path,start=start,stop=stop,step=step)

                        if save:
                            mapfile = f'dmaps_std/dmap_{name}_{param_string}.npy'
                            np.save(mapfile, dmaps_std)

    def calc_dmap_std(self, name, path, start=10,stop=None,step=1):
        # path = f'{self.sim_folder}'
        u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
        ag = u.atoms
        
        dmaps = []

        for t, ts in enumerate(u.trajectory[start:stop:step]):#,total=nframes):
            dmaps.append(cal.analysis.calc_dmap(ag,ag))
        dmaps = np.array(dmaps)
        dmaps_std = np.std(dmaps,axis=0)
        return dmaps_std
    
    def calc_dmap_std_manual(self, name, path, start=10,stop=None,step=1):
        # path = f'{self.sim_folder}'
        u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
        ag = u.atoms

        box = u.dimensions[:3] / 10.
        coordinates = u.trajectory.timeseries(ag) / 10.
        
        nres = len(ag)
        
        dmap_std_manual = self.calc_dmap_std_numba(coordinates,box,nres,start=start,stop=stop,step=step)
        return dmap_std_manual

    @staticmethod
    @nb.jit(nopython=True)
    def calc_dmap_std_numba(coordinates,box,nres,start=10,stop=None,step=1):
        dmaps_std_manual = np.zeros((nres,nres))
        
        for idx in range(nres):
            for jdx in range(idx,nres):
                x0 = coordinates[idx,start:stop:step]
                x1 = coordinates[jdx,start:stop:step]
                dvec = x1-x0
                dvec_pbc = np.where(dvec<=box/2, dvec, box-dvec)
                d = np.zeros((len(x0)))
                for kdx, dv_pbc in enumerate(dvec_pbc):
                    d[kdx] = math.sqrt(dv_pbc[0]**2 + dv_pbc[1]**2 + dv_pbc[2]**2)
                d_std = np.std(d)
        
                dmaps_std_manual[idx,jdx] = d_std
                dmaps_std_manual[jdx,idx] = d_std
        return dmaps_std_manual

    def load_dmap(self,name,k_go=0.,pae_shift=0.,pae_width=0.):
        param_string = self.get_param_string(k_go=k_go,pae_shift=pae_shift,pae_width=pae_width)
        mapfile = f'dmaps_std/dmap_{name}_{param_string}.npy'
        dmaps_std = np.load(mapfile)
        return dmaps_std

    def load_pae(self,name):
        input_pae = f'{self.pdb_folder}/{name}.json'
        pae = cal.build.load_pae(input_pae,symmetrize=True,colabfold=self.colabfold) / 10. # nm
        return pae

    def load_plddt(self,name):
        # pLDDT
        pdb = f'{self.pdb_folder}/{name}.pdb'
        bfac = cal.build.bfac_from_pdb(pdb)
        bfac_matrix = np.minimum.outer(bfac,bfac)
        return bfac_matrix

    def calc_restraints(self, name, k_go=0, pae_shift=0, pae_width=0,
                       min_scale=0.05):
        comp = cal.cfg.Components(
            fresidues = 'residues_CALVADOS3.csv',
            pdb_folder = self.pdb_folder,
            restraint = True,
            restraint_type = 'go',
        
            bfac_shift = self.bfac_shift, # 0.8, # 0.75,
            bfac_width = self.bfac_width, # 100., #30.,
            cutoff_restraint = self.cutoff_restraint,

            k_go = k_go,
            pae_shift = pae_shift, #0.6, # 0.3,
            pae_width = pae_width,# 15. # 15.,
        
            colabfold = self.colabfold,
        )
        
        comp.reset_components()
        comp.add(name=name)
        
        comp_dict = comp.components['system'][name]
        comp_defaults = comp.components['defaults']
        
        prot = cal.components.Protein(name, comp_dict, comp_defaults)
        
        prot.calc_properties()
        # idr, idr_length = self.calc_idr(prot.scale)
        
        prot.scale = np.where(prot.scale > min_scale, prot.scale, 0)
        prot.scale = np.where(prot.dmap < self.cutoff_restraint, prot.scale, 0)
        
        for idx in range(len(prot.scale)):
            prot.scale[idx,idx] = 0.

        return prot.scale

    @staticmethod
    @nb.jit(nopython=True)
    def flatten_array(x):
        xflat = []
        for idx in range(len(x)):
            for jdx in range(len(x)):
                if idx != jdx:
                    xflat.append(x[idx,jdx])
        xflat = np.array(xflat)
        return xflat

    def calc_pae_loss(self, min_key=None, max_key=None,
                      nsubsample=100, nbins=10):
        
        shape = (len(self.pae_widths), len(self.k_gos), len(self.pae_shifts))#, nbootstraps)
        self.spearman, self.pearson = np.zeros(shape), np.zeros(shape)
        self.spearman_err, self.pearson_err = np.zeros(shape), np.zeros(shape)
        
        for pw_idx, pae_width in enumerate(self.pae_widths):
            for k_idx, k_go in enumerate(self.k_gos):
                for ps_idx, pae_shift in enumerate(self.pae_shifts):
                    print(pae_width, k_go, pae_shift)
        
                    ps = []
                    sps = []
                    
                    for d_idx, (key, val) in enumerate(self.df[min_key:max_key].iterrows()):
                        name = val['uniprot']
                        pae = self.load_pae(name)
                        pae_flat = pae.flatten()
                        # pae_flat = paes_flat[d_idx]

                        edges = np.linspace(np.min(pae_flat), np.max(pae_flat), num=nbins+1)
                        # print(edges)
                        
                        dmaps_std = self.load_dmap(name, pae_width=pae_width, k_go=k_go, pae_shift=pae_shift)
                        dmaps_std_flat = dmaps_std.flatten()
                        
                        p_bins = []
        
                        for e_idx in range(nbins):
                            if e_idx == nbins-1:
                                # p_bin = np.argwhere((pae_flat>=edges[e_idx])).flatten()
                                p_bin = np.argwhere((pae_flat>=edges[e_idx]) & (pae_flat<=edges[e_idx+1])).flatten()
                            else:
                                p_bin = np.argwhere((pae_flat>=edges[e_idx]) & (pae_flat<edges[e_idx+1])).flatten()
                            p_bins.append(p_bin)
                        # print(len(p_bins))
                        
                        # nsam = np.min([len(p_bin) for p_bin in p_bins])
                        # nsam = np.min([1000, nsam])
                        nsam = 1000
        
                        p = []
                        sp = []
                        
                        for b_idx in range(nsubsample):
                            p_selects = np.array([],dtype=int)
                            for p_idx, p_bin in enumerate(p_bins):
                                p_select = np.random.choice(p_bin, size=nsam, replace=True) # replace or not?
                                p_selects = np.concatenate((p_selects, p_select))
            
                            pae_flat_selects = pae_flat[p_selects]
                            dmaps_std_flat_selects = dmaps_std_flat[p_selects]
            
                            p.append(pearsonr(pae_flat_selects,dmaps_std_flat_selects).statistic)
                            sp.append(spearmanr(pae_flat_selects,dmaps_std_flat_selects).statistic)
        
                        p = np.mean(p) # mean pearson per protein PAE subsampling
                        sp = np.mean(sp)
        
                        ps.append(p)
                        sps.append(sp)
        
                    metrics = np.array([ps,sps]).T # (nprots, 2)
                    metrics_m, errors = self.bootstrap_pae_loss_error(metrics)
        
                    self.pearson[pw_idx, k_idx, ps_idx] = metrics_m[0]
                    self.spearman[pw_idx, k_idx, ps_idx] = metrics_m[1]
        
                    self.pearson_err[pw_idx, k_idx, ps_idx] = errors[0]
                    self.spearman_err[pw_idx, k_idx, ps_idx] = errors[1]

    def bootstrap_pae_loss_error(self, metrics, n_bootstraps=1000):
        """ sensitivity, specificity, precision, accuracy """
        # print(metrics.shape) # requires shape=(len(prot), X)

        metrics_m = np.mean(metrics, axis=0)
        # print(metrics_m.shape)
        errors = np.zeros((n_bootstraps, metrics.shape[1])) # 4
        
        for idx in range(n_bootstraps):
            xs = np.arange(len(metrics),dtype=int)
            p = np.random.choice(xs, size=len(xs), replace=True)
            errors[idx] = np.mean(metrics[p], axis=0)

        errors = np.std(errors, axis=0) # along bootstraps

        return metrics_m, errors

    @staticmethod
    def calc_rg_metric(rg, rg_exp, rg_exp_err):
        chi2 = np.mean((rg-rg_exp)**2 / rg_exp_err**2)
        rmsd = np.sqrt(np.mean((rg-rg_exp)**2))
        rel = np.mean((rg-rg_exp) / rg_exp) * 100
        return chi2, rmsd, rel        
    
    def bootstrap_rg_metrics(self, rg, rg_exp, rg_exp_err, n_bootstraps=1000):
        """ chi2, rmsd, rel_error """
        chi2s_m, rmsds_m, rels_m = self.calc_rg_metric(rg, rg_exp, rg_exp_err)
        metrics = np.array([chi2s_m, rmsds_m, rels_m])

        errors = np.zeros((n_bootstraps, 3))

        for idx in range(n_bootstraps):
            xs = np.arange(len(rg),dtype=int)
            p = np.random.choice(xs, size=len(xs), replace=True)
            errors[idx] = self.calc_rg_metric(rg[p], rg_exp[p], rg_exp_err[p])

        errors = np.std(errors, axis=0)
        
        return metrics, errors

    def calc_rg_metrics(self, min_key=None, max_key=None):

        rg_exp = self.df['expRg']
        rg_exp_err = self.df['expRgErr']  

        shape = (len(self.pae_widths), len(self.k_gos), len(self.pae_shifts))
        self.rg_chi2s = np.zeros(shape)
        self.rg_chi2_errs = np.zeros(shape)

        self.rg_rmsds = np.zeros(shape)
        self.rg_rmsd_errs = np.zeros(shape)

        self.rg_rels = np.zeros(shape)
        self.rg_rel_errs = np.zeros(shape)

        for pw_idx, pae_width in enumerate(self.pae_widths):
            for k_idx, k_go in enumerate(self.k_gos):
                print(pae_width, k_go)
                for ps_idx, pae_shift in enumerate(self.pae_shifts):
                    rg = self.rgs[pw_idx,k_idx,ps_idx]
                    metrics, errors = self.bootstrap_rg_metrics(rg, rg_exp, rg_exp_err)
                    self.rg_chi2s[pw_idx,k_idx,ps_idx] = metrics[0]
                    self.rg_chi2_errs[pw_idx,k_idx,ps_idx] = errors[0]

                    self.rg_rmsds[pw_idx,k_idx,ps_idx] = metrics[1]
                    self.rg_rmsd_errs[pw_idx,k_idx,ps_idx] = errors[1]

                    self.rg_rels[pw_idx,k_idx,ps_idx] = metrics[2]
                    self.rg_rel_errs[pw_idx,k_idx,ps_idx] = errors[2]

    def plot_loss_vs_parameters(self,
                 metrics, errors, xlabel, ylabels, ylims, 
                savefile=None,
                cmap = plt.cm.Greens):
        
        fig, ax = plt.subplots(len(metrics),1,figsize=(4,1.*len(metrics)),sharex=True)
        
        for m_idx, (metric, error) in enumerate(zip(metrics,errors)):
            tlabels = []
            xs = []
            axij = ax[m_idx]
        
            for ps_idx, pae_shift in enumerate(self.pae_shifts): # pae_shifts
                for k_idx, k_go in enumerate(self.k_gos): # k_gos
                    x = 0.6*ps_idx+0.1*k_idx
                    axij.errorbar(x, metric[k_idx,ps_idx], yerr=error[k_idx,ps_idx], fmt='o', color=cmap((k_idx+1)/(len(self.k_gos))))
                    xs.append(x)
                    tlabels.append(k_go)
            axij.set(ylim=ylims[m_idx])
            # axij.set(xlim=(-0.5,xs[-1]+0.5))
            axij.set(ylabel=ylabels[m_idx])
            axij.set_xticks(xs)
            axij.set_xticklabels(tlabels)#,rotation=90)
            axij.grid(alpha=0.2)
            
            # for vline in np.arange(len(pae_shifts)-0.5, k_go*len(pae_shifts)-0.5,len(pae_shifts)):
                # axij.axvline(vline,ls='dashed',color='black', lw=0.5)
        
        # ax[0].hlines(45,xs[0],xs[len(rgs)-1])
            # fig.add_artist(lines.Line2D([0.14+xs[0],0.14+xs[len(rgs)-1]], [0.9, 0.9], linewidth=1,color='black'))
        ax[-1].set_xlabel(xlabel)#_\mathrm{Go}$')
        ax[-1].axhline(0,ls='dashed',lw=0.5,color='black')
        fig.tight_layout(h_pad=0.)
        if savefile is not None:
            fig.savefig(savefile)

# Select dataset

In [ ]:
# This requires the pdb data folder pdbs_df_30, pdbs_rg_set, and pdbs_zn to be in the current directory.

k_gos=[5,10,15,20,25] # epsilon
pae_shifts=[0.1,0.2,0.3,0.4,0.5] # beta_PAE
pae_widths=[15,30] # alpha_PAE

recalculate = False
step = 10 # step = 1 in paper

# This requires the simulation data folders sims_df_30, sims_rg_set, sims_zn to be in the current directory.
# Alternatively, the folders 'rgs' and 'dmaps_std' can be downloaded from zenodo and recalculate set to False

# Set with Rg experimental data
mdp_rg = MDP('rg_set',csv_file='mdp_rg.csv',colabfold=1,
                k_gos=k_gos,
                pae_shifts=pae_shifts,
                pae_widths=pae_widths,
            pdb_folder = 'pdbs_rg_set', # point to zenodo folder
            sim_folder = 'sims_rg_set', # point to zenodo folder
            )

# Set of 30 random MDPs from cytosol set
mdp_df_30 = MDP('df_30',csv_file='mdp_df_30.csv',colabfold=0,
                k_gos=k_gos,
                pae_shifts=pae_shifts,
                pae_widths=pae_widths,
            pdb_folder = 'pdbs_df_30', # point to zenodo folder
            sim_folder = 'sims_df_30', # point to zenodo folder
               )
                
# Set of 3 Zn finger-containing proteins
mdp_zn = MDP('zn',csv_file='mdp_zn.csv',colabfold=0,
                k_gos=k_gos,
                pae_shifts=pae_shifts,
                pae_widths=pae_widths,
            pdb_folder = 'pdbs_zn', # point to zenodo folder
            sim_folder = 'sims_zn', # point to zenodo folder
            )

if recalculate:
    # This takes some time to run. Only needs to run once.
    os.makedirs('rgs',exist_ok=True)
    mdp_rg.calc_rgs(step=step)

    os.makedirs('dmaps_std',exist_ok=True)
    mdp_df_30.calc_dmaps(step=step,manual=True)
    mdp_zn.calc_dmaps(step=step,manual=True)

mdp_rg.load_rgs()
mdp_rg.calc_rg_metrics()
mdp_df_30.calc_pae_loss(nsubsample=100)

In [ ]:
# Subset for some main figure plots
mdp_df_30_subset = MDP('df_30',csv_file='mdp_df_30.csv',colabfold=0,
                k_gos=[10,15,],
                pae_shifts=[0.1,0.2,0.3,0.4],
                pae_widths=[15],
                pdb_folder = 'pdbs_df_30', # point to zenodo folder
                sim_folder = 'sims_df_30', # point to zenodo folder)
)
mdp_df_30_subset.calc_pae_loss(nsubsample=100)

# PAE analysis

## PAE dmaps correlations unbalanced

In [ ]:
def bin_data(xs,ys,nbins,drange=None):
    """ bin data ys in xs bins, based on numpy.histogram_bin_edges """
    if drange == None:
        xmin, xmax = np.min(xs), np.max(xs)
    else:
        xmin, xmax = drange[0], drange[1]
    bins = np.linspace(xmin,xmax,nbins+1)
    y_binned = [[] for _ in range(nbins)]
    for x, y in zip(xs,ys):
        if x <= bins[0]:
            y_binned[0].append(y)
        elif x >= bins[-1]:
            y_binned[nbins-1].append(y)
        else:
            for idx in range(nbins):
                if x >= bins[idx] and x < bins[idx+1]:
                    y_binned[idx].append(y)
    return bins, y_binned

def mean_binned(xs,ys,nbins,drange=None):
    bins, y_binned = bin_data(xs,ys,nbins,drange=drange)
    y_m = np.array([np.mean(yb) for yb in y_binned])
    y_std = np.array([np.std(yb) for yb in y_binned])
    y_e = (bins[:-1] + bins[1:]) / 2
    return y_m, y_std, y_e

In [ ]:
mdp = copy.deepcopy(mdp_df_30) # or mdp_zn

min_key = 0
max_key = 30

cmap = plt.cm.Blues_r
vmin, vmax = 0., 3.

nprots = max_key-min_key

pae_width = 15

k_go = 15 # epsilon # Compare with 5
pae_shift = 0.3 # beta_PAE # Compare with 0.1

extra = 1 if nprots%10 > 0 else 0
fig, ax = plt.subplots(nprots//10+extra, 10, figsize=(10*1.3, (nprots//10+extra)*1.3))
for d_idx, (key, val) in enumerate(mdp.df[min_key:max_key].iterrows()):
    axij = ax[d_idx//10,d_idx%10]
    name = val['uniprot']
    pae = mdp.load_pae(name)
    pae_flat = pae.flatten()
    # pae_flat = paes_flat[d_idx]
    dmaps_std = mdp.load_dmap(name, pae_width=pae_width, k_go=k_go, pae_shift=pae_shift)
    dmaps_std_flat = dmaps_std.flatten()

    p = pearsonr(pae_flat,dmaps_std_flat).statistic
    sp = spearmanr(pae_flat,dmaps_std_flat).statistic

    # ax[d_idx].plot(pae_flat, dmaps_std_flat,'.',color='gray',alpha=0.3, markersize=1)
    H, x_e, y_e = np.histogram2d(pae_flat, dmaps_std_flat, bins=30, density=True)
    x_e = (x_e[:-1] + x_e[1:]) / 2.
    y_e = (y_e[:-1] + y_e[1:]) / 2.
    axij.imshow(H.T,
                extent=[x_e[0], x_e[-1], y_e[0], y_e[-1]],
                     vmin=0, vmax=0.3,
                cmap=plt.cm.Blues, origin='lower', aspect='auto')
    axij.set(xlabel='PAE [nm]', ylabel=r'$\sigma(r)$ [nm]')

    y_m, y_std, y_e = mean_binned(pae_flat, dmaps_std_flat, nbins=20)
    axij.errorbar(y_e, y_m, yerr=y_std, capsize=2, marker='o', color='C1', lw=0.7, markersize=1.5)
    axij.set_title(name,fontsize=6)
# fig.savefig(f'figures/pae_comparison/corr_unbalanced_{mdp.dataset}_pw{pae_width:.2f}_k{k_go:.2f}_ps{pae_shift:.2f}.pdf')

## PAE vs dmaps_std raw scan

In [ ]:
mdp = copy.deepcopy(mdp_df_30) # or mdp_df_30_subset

# Set min_key and max_key to None to get figures for all proteins
min_key = 0 # First protein
max_key = 5 # Last protein (exclusive)

pae_width = 15 # alpha_PAE
pw_idx = mdp.pae_widths.index(pae_width)
print(pw_idx, pae_width)

if len(mdp.k_gos) == 2:
    figtype = 'main'
else:
    figtype = 'SI'

nprots = max_key-min_key+1

for d_idx, (key, val) in tqdm(enumerate(mdp.df[min_key:max_key].iterrows()),total=nprots):

    name = val['uniprot']
    pae = mdp.load_pae(name)

    # fig, ax = plt.subplots(len(mdp.k_gos), len(mdp.pae_shifts)+1, figsize=((len(mdp.pae_shifts)+1)*2,len(mdp.k_gos)*2))
    fig, ax = plt.subplots(len(mdp.k_gos), len(mdp.pae_shifts)+1, figsize=((len(mdp.pae_shifts)+1)*1.5,len(mdp.k_gos)*1.3))
    
    for k_idx, k_go in enumerate(mdp.k_gos):
        if k_idx == 0:
            axij = ax[k_idx,0]
            plotmat(axij,pae,cmap=plt.cm.Oranges_r,vmin=0,vmax=3)
            # axij.set_title(f'{name}\nPAE', fontsize=6, linespacing=1.5)
            axij.set_title(f'PAE', fontsize=6, linespacing=1.5)
            axij.set(xlabel='Residue',ylabel='Residue')
            axij.grid(False)
        else:
            ax[k_idx,0].set_visible(False)
            ax[k_idx,1].set(ylabel='Residue')

        for ps_idx, pae_shift in enumerate(mdp.pae_shifts):
            axij = ax[k_idx,ps_idx+1]
            
            dmaps_std = mdp.load_dmap(name,k_go=k_go,pae_width=pae_width,pae_shift=pae_shift)

            plotmat(axij,dmaps_std,cmap=plt.cm.Blues_r,vmin=0,vmax=2.)
            axij.set_title(r'$\epsilon$:'+f'{k_go:.0f} kJ/mol  |  ' + r'$\beta_\mathrm{PAE}$:'+f'{pae_shift:.1f} nm', fontsize=6, linespacing=1.5)
            axij.set(xlabel='Residue')
            axij.grid(False)

    fig.tight_layout(h_pad=1)
    # fig.savefig(f'figures/pae_comparison/raw_maps/pae_vs_dmaps_raw_{name}_{figtype}.pdf')

## PAE vs dmaps_std opti

In [ ]:
mdp = copy.deepcopy(mdp_df_30) # or mdp_zn

dataset = 'df_30' #'zn'

min_key = 0
max_key = None

vmin, vmax = 0, 2

k_go = 15. # epsilon
pae_shift = 0.3 # beta_PAE
pae_width = 15. # alpha_PAE

nprots = len(mdp.df[min_key:max_key])

print(f'nprots: {nprots}')

if dataset == 'zn':
    prots_per_row = 3
    # figx = prots_per_row*4
    # figy = 3*max(1,nprots//prots_per_row)
    figx = prots_per_row*3
    figy = 2.5*max(1,nprots//prots_per_row)
else:
    prots_per_row = 4
    # figx = prots_per_row*4
    # figy = 2.2*max(1,nprots//prots_per_row)
    figx = prots_per_row*3
    figy = 2.*3/4*max(1,nprots//prots_per_row)
    
fig, ax = plt.subplots(nprots//prots_per_row+1,prots_per_row*2,figsize=(figx,figy))

nplots=(nprots//prots_per_row+1) * prots_per_row*2
print(f'nplots: {nplots}')

empty = nplots - 2*nprots
print(empty)

for d_idx, (key, val) in tqdm(enumerate(mdp.df[min_key:max_key].iterrows()),total=nprots):

    name = val['uniprot']
    pae = mdp.load_pae(name)

    dmaps_std = mdp.load_dmap(name,k_go=k_go,pae_width=pae_width,pae_shift=pae_shift)

    idx = d_idx//prots_per_row
    jdx = d_idx%prots_per_row
    
    if nprots < prots_per_row:
        axij_pae = ax[2*jdx]
        axij_dmaps = ax[2*jdx+1]
    else:
        axij_pae = ax[idx,2*jdx]
        axij_dmaps = ax[idx,2*jdx+1]
    
    _ = axij_pae.imshow(pae,cmap=plt.cm.Oranges_r,vmin=0,vmax=3.)

    divider = make_axes_locatable(axij_pae)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    axij_pae.set_title(f'{name}\nPAE', fontsize=6, linespacing=1.5)

    nres = len(pae)
    xs = np.arange(0,nres,100)

    cmap = plt.cm.Blues_r

    _ = axij_dmaps.imshow(dmaps_std,cmap=cmap,vmin=vmin,vmax=vmax)
    axij_dmaps.set_title(f'{name}\n'+'$\sigma(r)$ Sim.', fontsize=6, linespacing=1.5)
    divider = make_axes_locatable(axij_dmaps)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    for a in [axij_pae, axij_dmaps]:
        a.grid(False)
        # a.set(xlabel='Residue')#,ylabel='Residue')

for idx in range(1,empty+1):
    if nprots < prots_per_row:
        ax[-idx].set_visible(False)
    else:
        ax[-1,-idx].set_visible(False)
        ax[-2,-idx].set(xlabel='Residue')

for idx in range(len(ax)):
    ax[idx,0].set(ylabel='Residue')
    ax[-1,idx].set(xlabel='Residue')

fig.tight_layout(w_pad=0.,h_pad=1)
# fig.savefig(f'figures/pae_comparison/pae_vs_dmaps_std_opti_{mdp.dataset}.pdf')

# Rg analysis

## Rg scan raw vs exp

In [ ]:
mdp = copy.deepcopy(mdp_rg)

min_key = None
max_key = None

xs = np.arange(1.5,7.5,0.1)

fig, ax = plt.subplots(len(mdp.k_gos),len(mdp.pae_shifts),figsize=(8,8))#,figsize=(9,8))

pae_width = 15 # alpha_PAE
pw_idx = mdp.pae_widths.index(pae_width)
print(pw_idx, pae_width)

labels = ['AF-CALVADOS','CALVADOS 3']

for k_idx, k_go in enumerate(mdp.k_gos):
    for ps_idx, pae_shift in enumerate(mdp.pae_shifts):
        axij = ax[k_idx,ps_idx]
        axij.plot(mdp.df['expRg'][min_key:max_key],mdp.rgs[pw_idx,k_idx,ps_idx][min_key:max_key],'o',label=labels[0])
        axij.plot(mdp.df['expRg'][min_key:max_key],mdp.df['cal'][min_key:max_key],'x',color='C1',label=labels[1])
        axij.plot(xs,xs,lw=0.5,color='black')
        axij.set(xlabel=r'$R_g$ Experiment', ylabel=r'$R_g$ Simulation')
        axij.set_title(r'$\epsilon$: '+f'{k_go:.0f} kJ/mol  |  '+r'$\beta_\mathrm{PAE}$: '+f'{pae_shift:.1f} nm',fontsize=6)
        axij.set(xlim=(xs[0],xs[-1]))
# ax[0,0].legend()
ax[0,0].text(4.5,8.1,'AF-CALVADOS',color='C0',fontsize=6)
ax[0,0].text(4,3,'CALVADOS 3',color='C1',fontsize=6)
fig.tight_layout()
# fig.savefig(f'figures/rgs/rg_raw_{mdp.dataset}_pae_width_{pae_width:.2f}.pdf')

## Rg scan loss vs parameters

In [ ]:
mdp = copy.deepcopy(mdp_rg)

pae_width = 15
pw_idx = mdp.pae_widths.index(pae_width)
print(pw_idx, pae_width)

xlabel = r'$\epsilon$ [kJ/mol]'
ylabels = [r'$\chi^2$','RMSD [nm]','Relative error [%]']
ylims = [
    (0,60),
    (0.1, 0.5),
    (-5, 10)
]

metrics = [mdp.rg_chi2s[pw_idx], mdp.rg_rmsds[pw_idx], mdp.rg_rels[pw_idx]]
errors = [1.96*mdp.rg_chi2_errs[pw_idx], 1.96*mdp.rg_rmsd_errs[pw_idx], 1.96*mdp.rg_rel_errs[pw_idx]]

mdp.plot_loss_vs_parameters(metrics, errors, xlabel, ylabels, ylims)#, 
                # savefile=f'figures/rgs/rg_statistics_{dataset}_pae_width_{pae_width:.2f}.pdf')

## Rg opti raw and rel

In [ ]:
mdp = copy.deepcopy(mdp_rg)

min_key = None
max_key = None

xs = np.arange(1.5,7.5,0.1)

pae_width = 15 # alpha_PAE
k_go = 15. # epsilon
pae_shift = 0.3 # beta_PAE

pw_idx = mdp.pae_widths.index(pae_width)
k_idx = mdp.k_gos.index(k_go)
ps_idx = mdp.pae_shifts.index(pae_shift)

print(pw_idx, k_idx, ps_idx)

rg_sim_fan = mdp.df['cal'][min_key:max_key]
rg_sim = mdp.rgs[pw_idx,k_idx,ps_idx]
rg_exp = mdp.df['expRg'][min_key:max_key]
rg_exp_err = mdp.df['expRgErr'][min_key:max_key]

rel_signed_err = (rg_sim - rg_exp) / rg_exp * 100
rel_signed_err_fan = (rg_sim_fan - rg_exp) / rg_exp * 100
rel_exp_err = rg_exp_err / rg_exp * 100

chi2 = np.mean((rg_sim - rg_exp)**2 / rg_exp_err**2)

print(chi2, mdp.rg_chi2s[pw_idx,k_idx,ps_idx])

chi2_fan = np.mean((rg_sim_fan - rg_exp)**2 / rg_exp_err**2)

labels = ['AF-CALVADOS','CALVADOS 3']

c3_color = 'skyblue'
cgo_color = 'tab:green'

fig, ax = plt.subplots(1,2,figsize=(col2x, .7*col1x),width_ratios=(0.25,0.75))

axij = ax[0]#[idx,jdx]
axij.plot(rg_exp,rg_sim_fan,'o',color=c3_color,label=labels[1],fillstyle='none')
axij.plot(rg_exp,rg_sim,'o',label=labels[0],color=cgo_color)

axij.plot(xs,xs,lw=0.5,color='black')
axij.set_xticks(np.arange(2,7.1))
axij.set_yticks(np.arange(2,7.1))
axij.set(xlabel=r'$R_g$ Experiment [nm]', ylabel=r'$R_g$ Simulation [nm]')
# axij.set_title(f'k_go:{k_gos[idx]:.2f} | pae_shift:{pae_shifts[jdx]:.2f}')

axij.legend()

###

axij = ax[1]

N = len(rg_sim)

xs = np.arange(len(rel_signed_err))*2.5
axij.bar(xs, rel_signed_err_fan,
    label=f'CALVADOS 3, $\chi^2=${chi2_fan:.0f}' + r'$\pm 4$' ,color=c3_color,
       width=1.0)
axij.bar(xs+1, rel_signed_err,
    label=f'AF-CALVADOS, $\chi^2=${chi2:.0f}' + r'$\pm$' + f'{mdp.rg_chi2_errs[pw_idx,k_idx,ps_idx]:.0f}',color=cgo_color,
       width=1.0)

axij.errorbar(xs+0.5,np.zeros(N),yerr=rel_exp_err, c='black',ls='none',capsize=4,alpha=1.)

axij.set_xticks(xs+0.5)
axij.set_xticklabels(mdp.df['uniprot'][min_key:max_key],rotation=90)

axij.set(xlim=(xs[0]-1,xs[-1]+2))
axij.set(ylabel=r'($R_{g,\mathrm{pred}}$ - $R_{g,\mathrm{exp}}$) / $R_{g,\mathrm{exp}}$ %')
axij.legend(loc='upper right')
axij.grid(axis='x')#alpha=0.3)#ydata=[])
# axij.set_title(f'pae_width: {pae_width:.1f} | k_go: {k_gos[idx]:.1f} | pae_shift: {pae_shifts[jdx]:.1f}')

fig.tight_layout()
# fig.savefig(f'figures/rgs/rg_opti_raw.pdf')

# Other paper figures

## Fig. 1

In [ ]:
k_go = 15. # epsilon
pae_shift = 0.3 # beta_PAE
pae_width = 15. # alpha_PAE
bfac_width = 50 # alpha_pLDDT
bfac_shift = 0.8 # beta_pLDDT

cutoff_restraint = 0.9 # nm

nprots = len(mdp_df_30.df[min_key:max_key])

print(f'nprots: {nprots}')

prots_per_row = 4

name = 'P50570'
colabfold = 0

fig, ax = plt.subplots(2,2,figsize=(1.*col1x,0.7*col1x))

axij_plddt = ax[0,0]
axij_pae = ax[1,0]
axij_restr = ax[0,1]
axij_dmaps = ax[1,1]

# pLDDT
pdb = f'{mdp_df_30.pdb_folder}/{name}.pdb'
bfac = cal.build.bfac_from_pdb(pdb)
bfac_matrix = np.minimum.outer(bfac,bfac)

plotmat(axij_plddt,bfac_matrix,cmap=plt.cm.Greys,vmin=0.6,vmax=1.,unitlabel=None)
axij_plddt.set_title('pLDDT',fontsize=6)

# PAE
input_pae = f'{mdp_df_30.pdb_folder}/{name}.json'
pae = cal.build.load_pae(input_pae,symmetrize=True,colabfold=colabfold) / 10. # nm

plotmat(axij_pae,pae,cmap=plt.cm.Oranges_r,vmin=0,vmax=3.)
axij_pae.set_title('PAE', fontsize=6, linespacing=1.5)

# restraints
comp = cal.cfg.Components(
    fresidues = 'residues_CALVADOS3.csv',
    pdb_folder = mdp_df_30.pdb_folder,
    restraint = True,
    restraint_type = 'go',

    k_go = k_go,
    bfac_shift = bfac_shift, # 0.8, # 0.75,
    bfac_width = bfac_width, # 100., #30.,
    pae_shift = pae_shift, #0.6, # 0.3,
    pae_width = pae_width,# 15. # 15.,

    cutoff_restraint = cutoff_restraint,

    colabfold = colabfold,
)

comp.reset_components()
comp.add(name=name)

comp_dict = comp.components['system'][name]
comp_defaults = comp.components['defaults']

prot = cal.components.Protein(name, comp_dict, comp_defaults)

prot.calc_properties()
idr, idr_length = calc_idr(prot.scale)

prot.scale = np.where(prot.scale > 0.05, prot.scale, 0)
prot.scale = np.where(prot.dmap < 0.9, prot.scale, 0)


for idx in range(len(prot.scale)):
    prot.scale[idx,idx] = 0.

plotmat(axij_restr,prot.scale,cmap=plt.cm.Greens,vmin=0,vmax=0.2,unitlabel=None)
axij_restr.set_title('Restraints', fontsize=6)

# dmaps_std
mapfolder = f'dmaps_std/dmap_{name}_k_{k_go:.2f}_cutoff_{cutoff_restraint:.2f}_bshift_{bfac_shift:.2f}_bwidth_{bfac_width:.2f}_paeshift_{pae_shift:.2f}_paewidth_{pae_width:.2f}.npy'
dmaps_std = np.load(mapfolder)

plotmat(axij_dmaps,dmaps_std,cmap=plt.cm.Blues_r,vmin=0,vmax=3.)
axij_dmaps.set_title('$\sigma(r_{ij})$ Sim.', fontsize=6)

for idx in range(2):
    for jdx in range(2):
        ax[idx,jdx].grid(False)
        ax[idx,jdx].set(xlabel='Residue',ylabel='Residue')

# fig.savefig('figures/fig1_panelA.pdf')

# Fig. sigmoid

In [ ]:
sig_params = [
    (0.8, 50),
    (0.3, -15)
]

xs = [
    np.arange(0,1,0.01),
    np.arange(0,1.5,0.01)
]

xlabels = [r'pLDDT$_{ij}$', r'PAE$_{ij}$']
ylabels = [r'$\sigma$(pLDDT)', r'$\sigma$(PAE)']
    
fig, ax = plt.subplots(1,2,figsize=(.7*col1x,0.3*col1x))

for idx in range(2):
    axij = ax[idx]
    x = xs[idx]
    shift, width = sig_params[idx]
    y = np.exp(width*(x-shift)) / (np.exp(width*(x-shift)) + 1)
    
    axij.plot(x,y,color='black',lw=1)

    axij.set(xlabel=xlabels[idx], ylabel=ylabels[idx])
ax[1].spines['top'].set_visible(False)
ax[1].spines['right'].set_visible(False)

ax[0].grid(False)
ax[1].grid(False)

alpha = 0.5

ax[0].axvspan(0.,0.5,color=(254/255,125/255,69/255),alpha=alpha,ls='none') # yellow
ax[0].axvspan(0.5,0.7,color=(255/255,219/255,19/255),alpha=alpha,ls='none') # yellow
ax[0].axvspan(0.7,0.9,color=(88/255,143/255,255/255),alpha=alpha,ls='none') # light blue
ax[0].axvspan(0.9,1.,color=(0,70/255,214/255),alpha=alpha,ls='none') # dark blue

ax[0].axvline(0.8,color='black',ls='dashed',lw=0.8)
ax[0].text(0.78,0.9,r'$\beta_\mathrm{pLDDT}$',color='black',horizontalalignment='right')
ax[0].set(xlim=(xs[0][0],xs[0][-1]))

ax[1].axvline(0.3,color='black',ls='dashed',lw=0.8)
ax[1].text(0.33,0.9,r'$\beta_\mathrm{PAE}$',color='black')
ax[1].set(xlim=(xs[1][0],xs[1][-1]))

fig.tight_layout()
# fig.savefig('figures/sigmoids.pdf')

## Combined Rg and PAE scan results

In [ ]:
def combined_plot_loss_vs_parameters(
             metrics, errors, xlabel, ylabels, ylims, 
            savefile=None,
            cmap = plt.cm.Greens):
    
    fig, ax = plt.subplots(len(metrics),1,figsize=(4,0.9*len(metrics)),sharex=True)
    
    for m_idx, (metric, error) in enumerate(zip(metrics,errors)):
        tlabels = []
        xs = []
        axij = ax[m_idx]
    
        for ps_idx, pae_shift in enumerate(mdp_rg.pae_shifts): # pae_shifts
            for k_idx, k_go in enumerate(mdp_rg.k_gos): # k_gos
                x = 0.6*ps_idx+0.1*k_idx
                axij.errorbar(x, metric[k_idx,ps_idx], yerr=error[k_idx,ps_idx], fmt='o', color=cmap((k_idx+1)/(len(mdp_rg.k_gos))))
                xs.append(x)
                tlabels.append(k_go)
        axij.set(ylim=ylims[m_idx])
        # axij.set(xlim=(-0.5,xs[-1]+0.5))
        axij.set(ylabel=ylabels[m_idx])
        axij.set_xticks(xs)
        axij.set_xticklabels(tlabels)#,rotation=90)
        axij.grid(alpha=0.2)
        
        # for vline in np.arange(len(pae_shifts)-0.5, k_go*len(pae_shifts)-0.5,len(pae_shifts)):
            # axij.axvline(vline,ls='dashed',color='black', lw=0.5)
    
    # ax[0].hlines(45,xs[0],xs[len(rgs)-1])
        # fig.add_artist(lines.Line2D([0.14+xs[0],0.14+xs[len(rgs)-1]], [0.9, 0.9], linewidth=1,color='black'))
    ax[-1].set_xlabel(xlabel)#_\mathrm{Go}$')
    ax[-1].axhline(0,ls='dashed',lw=0.5,color='black')
    fig.tight_layout(h_pad=0.)
    if savefile is not None:
        fig.savefig(savefile)

In [ ]:
pae_width = 15
pw_idx = mdp_rg.pae_widths.index(pae_width)
print(pw_idx, pae_width)

xlabel = r'$\epsilon$ [kJ/mol]'
# ylabels = [r'$\chi^2 (R_\mathrm{g})$',r'Rel. $R_\mathrm{g}$ error [%]','PAE Sensitivity','PAE Specificity']#'RMSD [nm]','Relative error [%]']
ylabels = [r'$\chi^2 (R_\mathrm{g})$',r'Rel. $R_\mathrm{g}$ error [%]',r'Pearson $r$',r'Spearman $\rho$']#'RMSD [nm]','Relative error [%]']
ylims = [
    (-5,60),
    # (0.1, 0.5),
    (-5, 15),
    (0.7, None),
    (0.7, None),  
]

metrics = [mdp_rg.rg_chi2s[pw_idx], mdp_rg.rg_rels[pw_idx], 
           mdp_df_30.pearson[pw_idx], mdp_df_30.spearman[pw_idx]]

errors = [1.96*mdp_rg.rg_chi2_errs[pw_idx], 1.96*mdp_rg.rg_rel_errs[pw_idx], 
          1.96*mdp_df_30.pearson_err[pw_idx], 1.96*mdp_df_30.spearman_err[pw_idx]]

combined_plot_loss_vs_parameters(metrics, errors, xlabel, ylabels, ylims)#, 
                # savefile=f'figures/combined_statistics_pae_width_{pae_width:.2f}_new.pdf')